# 稀缺度与移民价值排行

Scarcity & Immigration Value Rankings — Which occupations are most needed in developed countries and most internationally mobile?

评分色阶：红色(低分0) → 黄色(中等5) → 绿色(高分10)

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Heiti TC', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

from pathlib import Path

csv_dir = Path('../data/csv')
all_files = sorted(csv_dir.glob('*.csv'))
print(f"Loading {len(all_files)} CSV files ...")

dfs = []
for f in all_files:
    tmp = pd.read_csv(f)
    dfs.append(tmp)
    print(f"  {f.name}: {len(tmp)} rows")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(df)} rows, {df['sub_category'].nunique()} occupations, "
      f"{df['country_or_region'].nunique()} countries/regions")


In [ ]:
# Key columns for scarcity / immigration analysis
base_cols = ['sub_category', 'sub_category_en', 'country_or_region', 'major_category', 'major_code', 'region']
imm_cols = ['developed_scarcity', 'intl_mobility', 'value_added', 'growth_coeff',
            'license_barrier', 'supply_demand', 'stability', 'composite_index']
display_cols = base_cols + imm_cols


## Top 50 Most Scarce Occupations in Developed Countries

In [ ]:
top50_scarce = df.nlargest(50, 'developed_scarcity')[display_cols].reset_index(drop=True)
top50_scarce.index = top50_scarce.index + 1
top50_scarce.index.name = 'Rank'

score_gradient = [c for c in imm_cols if c in df.columns]
styled = top50_scarce.style \
    .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## Top 50 Highest International Mobility

In [ ]:
top50_mobile = df.nlargest(50, 'intl_mobility')[display_cols].reset_index(drop=True)
top50_mobile.index = top50_mobile.index + 1
top50_mobile.index.name = 'Rank'

styled = top50_mobile.style \
    .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## Cross-Tab: Developed Scarcity by Major Category x Region

Mean developed_scarcity score for each combination.

In [ ]:
cross = pd.pivot_table(df, values='developed_scarcity',
                        index='major_category', columns='region',
                        aggfunc='mean').round(2)
cross = cross.fillna(0)

styled = cross.style \
    .background_gradient(cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## Immigration-Valuable Occupations

Occupations with BOTH high developed_scarcity (>= 7) AND high intl_mobility (>= 7) — the best candidates for skilled immigration pathways.

In [ ]:
imm_valuable = df[(df['developed_scarcity'] >= 7) & (df['intl_mobility'] >= 7)].copy()
imm_valuable = imm_valuable.sort_values('developed_scarcity', ascending=False)

print(f"Found {len(imm_valuable)} occupation-country combinations meeting criteria "
      f"(developed_scarcity >= 7 AND intl_mobility >= 7)\n")
print(f"Unique occupations: {imm_valuable['sub_category'].nunique()}")
print(f"Unique countries: {imm_valuable['country_or_region'].nunique()}")

imm_display = imm_valuable[display_cols].reset_index(drop=True)
imm_display.index = imm_display.index + 1
imm_display.index.name = 'Rank'

styled = imm_display.style \
    .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## Immigration Value Summary by Major Category

In [ ]:
imm_summary = imm_valuable.groupby('major_category').agg(
    count=('sub_category', 'size'),
    unique_occupations=('sub_category', 'nunique'),
    avg_scarcity=('developed_scarcity', 'mean'),
    avg_mobility=('intl_mobility', 'mean'),
    avg_composite=('composite_index', 'mean'),
).round(2).sort_values('count', ascending=False)

styled = imm_summary.style \
    .background_gradient(subset=['avg_scarcity', 'avg_mobility', 'avg_composite'], cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '12px'})
styled
